In [1]:
import os

MODE = "glue"  # Set to "databricks" if running outside Glue

# S3 bucket names — read from environment variables or defaults
S3_RAW_BUCKET       = os.environ.get("S3_RAW_BUCKET", "f1-pipeline-raw-layer-034381339055")
S3_PROCESSED_BUCKET = os.environ.get("S3_PROCESSED_BUCKET", "f1-pipeline-processed-layer-034381339055")
AWS_REGION          = os.environ.get("AWS_REGION", "ap-south-1")

# S3 scheme: Databricks uses s3a://, Glue uses s3://
S3_SCHEME      = "s3a" if MODE == "databricks" else "s3"
RAW_BASE       = f"{S3_SCHEME}://{S3_RAW_BUCKET}/raw/openf1"
PROCESSED_BASE = f"{S3_SCHEME}://{S3_PROCESSED_BUCKET}/processed/openf1"

# Manifest storage
MANIFEST_BUCKET = S3_RAW_BUCKET
MANIFEST_PREFIX = "meta/manifests"

print(f"MODE:      {MODE}")
print(f"RAW:       {RAW_BASE}")
print(f"PROCESSED: {PROCESSED_BASE}")

Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 1a903698-6b5c-4785-82f3-aec01b32eb72
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 1a903698-6b5c-4785-82f3-aec01b32eb72 to get into ready status...
Session 1a903698-6b5c-4785-82f3-aec01b32eb72 has been created.
MODE:      glue
RAW:       s3://f1-pipeline-raw-layer-034381339055/raw/openf1
PROCESSED: s3://f1-pipeline-processed-layer-034381339055/processed/openf1


In [2]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

# Initialize Glue context and Spark session
sc          = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark       = glueContext.spark_session

# Initialize Glue Job 
# (try/except allows this to run interactively in notebooks without --JOB_NAME)
job = Job(glueContext)
try:
    args = getResolvedOptions(sys.argv, ["JOB_NAME"])
    job.init(args["JOB_NAME"], args)
except Exception:
    job.init("f1-openf1-interactive-notebook", {})

spark.sparkContext.setLogLevel("WARN")
print("Spark & Glue session ready.")

Spark & Glue session ready.


In [3]:
import boto3
from botocore.exceptions import ClientError
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType, StringType, BooleanType

def seconds_to_ms_col(col_name):
    """Convert a DOUBLE column (seconds) to LONG (milliseconds)."""
    return (F.col(col_name).cast(DoubleType()) * 1000).cast(LongType())

def discover_session_paths():
    """Discover all year/meeting/session directories that contain data in S3."""
    paths, s3, seen = [], boto3.client("s3"), set()
    paginator = s3.get_paginator("list_objects_v2")
    
    for page in paginator.paginate(Bucket=MANIFEST_BUCKET, Prefix="raw/openf1/"):
        for obj in page.get("Contents", []):
            parts = obj["Key"].split("/")
            if len(parts) < 5:
                continue
            year_str, meeting_dir, session_type = parts[2], parts[3], parts[4]
            if not year_str.isdigit() or not meeting_dir.startswith("meeting_"):
                continue
            key = (year_str, meeting_dir, session_type)
            if key not in seen:
                seen.add(key)
                paths.append({
                    "year": int(year_str),
                    "meeting_key": meeting_dir.replace("meeting_", ""),
                    "session_type": session_type,
                    "path": f"{RAW_BASE}/{year_str}/{meeting_dir}/{session_type}",
                })
    return paths

def safe_read_json(path):
    """Try reading a JSON file. Return None if it doesn't exist or is empty."""
    try:
        df = spark.read.json(path)
        if df.head(1):
            return df
        return None
    except Exception:
        return None

In [4]:
import json
from datetime import datetime, timezone

_manifest_s3 = None

def _get_manifest_s3():
    """Lazy-init S3 client for manifest operations."""
    global _manifest_s3
    if _manifest_s3 is None:
        _manifest_s3 = boto3.client("s3")
    return _manifest_s3

def load_manifest(source):
    """Read manifest JSON from S3. Returns empty manifest if not found."""
    key = f"{MANIFEST_PREFIX}/{source}_manifest.json"
    try:
        obj = _get_manifest_s3().get_object(Bucket=MANIFEST_BUCKET, Key=key)
        return json.loads(obj["Body"].read())
    except ClientError as e:
        if e.response["Error"]["Code"] == "NoSuchKey":
            return {"last_updated": None, "processed": {}}
        raise

def save_manifest(source, manifest):
    """Write manifest dict back to S3 with updated timestamp."""
    manifest["last_updated"] = datetime.now(timezone.utc).isoformat()
    key = f"{MANIFEST_PREFIX}/{source}_manifest.json"
    _get_manifest_s3().put_object(
        Bucket=MANIFEST_BUCKET, Key=key,
        Body=json.dumps(manifest, indent=2), ContentType="application/json",
    )

def mark_file_done(s3_path, manifest):
    """Record an S3 path as successfully processed in the manifest dict."""
    manifest["processed"][s3_path] = datetime.now(timezone.utc).isoformat()

def get_unprocessed(source, all_raw_paths):
    """Return (unprocessed_paths, manifest) for the given source."""
    manifest = load_manifest(source)
    already = set(manifest["processed"].keys())
    unprocessed = [p for p in all_raw_paths if p not in already]
    print(f"  {source}: {len(already)} already processed, {len(unprocessed)} new")
    return unprocessed, manifest

In [5]:
def clean_drivers(df):
    cols_to_drop = [c for c in df.columns if c in ("headshot_url",)]
    df = df.drop(*cols_to_drop)
    return (
        df
        .withColumn("driver_number", F.col("driver_number").cast(IntegerType()))
        .withColumn("session_key",   F.col("session_key").cast(IntegerType()))
        .withColumn("meeting_key",   F.col("meeting_key").cast(IntegerType()))
        .dropDuplicates()
    )

def clean_laps(df):
    segment_cols = [c for c in df.columns if c.startswith("segments_sector")]
    df = df.drop(*segment_cols)
    return (
        df
        .withColumn("driver_number",    F.col("driver_number").cast(IntegerType()))
        .withColumn("session_key",      F.col("session_key").cast(IntegerType()))
        .withColumn("meeting_key",      F.col("meeting_key").cast(IntegerType()))
        .withColumn("lap_number",       F.col("lap_number").cast(IntegerType()))
        # Convert seconds → milliseconds
        .withColumn("lap_duration_ms",       seconds_to_ms_col("lap_duration"))
        .withColumn("duration_sector_1_ms",  seconds_to_ms_col("duration_sector_1"))
        .withColumn("duration_sector_2_ms",  seconds_to_ms_col("duration_sector_2"))
        .withColumn("duration_sector_3_ms",  seconds_to_ms_col("duration_sector_3"))
        # Keep original seconds as DOUBLE
        .withColumn("lap_duration",          F.col("lap_duration").cast(DoubleType()))
        .withColumn("duration_sector_1",     F.col("duration_sector_1").cast(DoubleType()))
        .withColumn("duration_sector_2",     F.col("duration_sector_2").cast(DoubleType()))
        .withColumn("duration_sector_3",     F.col("duration_sector_3").cast(DoubleType()))
        # Speed traps
        .withColumn("i1_speed",  F.col("i1_speed").cast(DoubleType()))
        .withColumn("i2_speed",  F.col("i2_speed").cast(DoubleType()))
        .withColumn("st_speed",  F.col("st_speed").cast(DoubleType()))
        .withColumn("is_pit_out_lap", F.col("is_pit_out_lap").cast(BooleanType()))
        .dropDuplicates()
    )

def clean_stints(df):
    return (
        df
        .withColumn("driver_number",      F.col("driver_number").cast(IntegerType()))
        .withColumn("session_key",        F.col("session_key").cast(IntegerType()))
        .withColumn("meeting_key",        F.col("meeting_key").cast(IntegerType()))
        .withColumn("stint_number",       F.col("stint_number").cast(IntegerType()))
        .withColumn("lap_start",          F.col("lap_start").cast(IntegerType()))
        .withColumn("lap_end",            F.col("lap_end").cast(IntegerType()))
        .withColumn("tyre_age_at_start",  F.col("tyre_age_at_start").cast(IntegerType()))
        .dropDuplicates()
    )

def clean_pit(df):
    return (
        df
        .withColumn("driver_number",  F.col("driver_number").cast(IntegerType()))
        .withColumn("session_key",    F.col("session_key").cast(IntegerType()))
        .withColumn("meeting_key",    F.col("meeting_key").cast(IntegerType()))
        .withColumn("lap_number",     F.col("lap_number").cast(IntegerType()))
        .withColumn("pit_duration_ms",
                    (F.col("pit_duration").cast(DoubleType()) * 1000).cast(LongType()))
        .withColumn("pit_duration",   F.col("pit_duration").cast(DoubleType()))
        .dropDuplicates()
    )

In [7]:
# Endpoint -> (clean function, JSON filename)
ENDPOINT_CONFIG = {
    "drivers":  (clean_drivers,  "drivers.json"),
    "laps":     (clean_laps,     "laps.json"),
    "stints":   (clean_stints,   "stints.json"),
    "pit":      (clean_pit,      "pit.json"),
}

print("=" * 60)
print("clean_openf1 — Starting")
print("=" * 60)

# 1. Discover paths
session_paths = discover_session_paths()
print(f"  Found {len(session_paths)} session paths to scan\n")

# 2. Build all expected raw S3 keys
ALL_RAW_PATHS = []
for sp in session_paths:
    for _, (_, filename) in ENDPOINT_CONFIG.items():
        ALL_RAW_PATHS.append(
            f"raw/openf1/{sp['year']}/meeting_{sp['meeting_key']}"
            f"/{sp['session_type']}/{filename}"
        )

# 3. Check manifest
unprocessed, manifest = get_unprocessed("openf1", ALL_RAW_PATHS)
if not unprocessed:
    print("All files already processed — nothing to do.")
else:
    # 4. Process files
    for raw_path in unprocessed:
        parts = raw_path.split("/")
        year         = int(parts[2])
        meeting_key  = parts[3].replace("meeting_", "")
        session_type = parts[4]
        filename     = parts[5]

        # Look up the matching cleaner function
        clean_fn = None
        endpoint_name = None
        for ep_name, (fn, fn_name) in ENDPOINT_CONFIG.items():
            if fn_name == filename:
                clean_fn = fn
                endpoint_name = ep_name
                break

        if clean_fn is None:
            print(f"  ⚠ Unknown file {filename} — skipping")
            continue

        json_path = f"{RAW_BASE}/{year}/meeting_{meeting_key}/{session_type}/{filename}"
        df_raw = safe_read_json(json_path)
        
        if df_raw is None:
            continue  # Don't mark as done — retry on next run

        df_clean = clean_fn(df_raw)
        
        # Use isEmpty() to avoid evaluating the whole dataframe just to check if it's empty
        if df_clean is None or df_clean.rdd.isEmpty():
            continue

        if "year" not in df_clean.columns:
            df_clean = df_clean.withColumn("year", F.lit(year).cast(IntegerType()))

        # Write with append mode (incremental per file)
        path = f"{PROCESSED_BASE}/{endpoint_name}"
        df_clean.write.mode("append").partitionBy("year").parquet(path)
        print(f"  ✓ {raw_path}")

        mark_file_done(raw_path, manifest)
        save_manifest("openf1", manifest)

    print("\n" + "=" * 60)
    print("clean_openf1 — Complete ✓")
    print("=" * 60)

# Commit the Glue job so AWS marks it as Succeeded
job.commit()

clean_openf1 — Starting
  Found 2 session paths to scan

  openf1: 0 already processed, 8 new
  ✓ raw/openf1/2026/meeting_1279/race/drivers.json
  ✓ raw/openf1/2026/meeting_1279/race/laps.json
  ✓ raw/openf1/2026/meeting_1279/race/stints.json
  ✓ raw/openf1/2026/meeting_1279/race/pit.json
  ✓ raw/openf1/2026/meeting_1280/race/drivers.json
  ✓ raw/openf1/2026/meeting_1280/race/laps.json
  ✓ raw/openf1/2026/meeting_1280/race/stints.json
  ✓ raw/openf1/2026/meeting_1280/race/pit.json

clean_openf1 — Complete ✓
